In [ ]:
  # Install
!pip install -q transformers accelerate psutil onnxruntime optimum

In [ ]:
# Base model (Unoptimized)

import torch
import time
import psutil
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-0.5B"

tokenizer = AutoTokenizer.from_pretrained(model_name)

baseline_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float32,
    device_map="cpu"
)

baseline_model.eval()

In [ ]:
# Optimized model (Deterministic mode,use_cache=True,Reduced max tokens,torch.compile,Smaller context input,Batching)

optimized_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float32,
    device_map="cpu",
    use_cache=True
)

optimized_model.eval()

# Optional: compile (PyTorch 2.x required)
try:
    optimized_model = torch.compile(optimized_model)
    print("Model compiled successfully")
except:
    print("Compilation not supported")

In [ ]:
# Benchmark function

def benchmark(model, prompt_list, max_new_tokens=100):
    inputs = tokenizer(
        prompt_list,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512  # reduce context length
    )

    process = psutil.Process()
    ram_before = process.memory_info().rss / 1024**2

    start = time.time()

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,        # deterministic
            use_cache=True
        )

    end = time.time()

    ram_after = process.memory_info().rss / 1024**2

    total_time = end - start
    tokens_generated = outputs.shape[1] - inputs["input_ids"].shape[1]
    tokens_per_sec = tokens_generated / total_time

    print("\n Results ")
    print(f"Time: {total_time:.2f} sec")
    print(f"Tokens/sec: {tokens_per_sec:.2f}")
    print(f"RAM used: {ram_after - ram_before:.2f} MB")

    return total_time, tokens_per_sec

In [ ]:
# Run Base Model

prompts = ["Explain AI in simple words."]

print("Running Base Model")
benchmark(baseline_model, prompts, max_new_tokens=100)

In [ ]:
# Run Optimized Model

print("Running Optimized Model")
benchmark(optimized_model, prompts, max_new_tokens=50)  # reduced tokens

In [ ]:
# Batch Small Prompts

batch_prompts = [
    "What is AI?",
    "Define machine learning.",
    "What is deep learning?"
]

print("Running Batched Optimized...")
benchmark(optimized_model, batch_prompts, max_new_tokens=50)

In [ ]:
!pip install optimum[onnxruntime]
!pip install onnxruntime

In [ ]:
# convert to onnx


from optimum.onnxruntime import ORTModelForCausalLM

onnx_model = ORTModelForCausalLM.from_pretrained(
    model_name,
    export=True
)

print("ONNX Model Ready")
print("Running ONNX")
benchmark(onnx_model, prompts, max_new_tokens=50)